### Backup 2: The DeepRitz-Method

While PINNs usually utilize the strong formulation of the PDE, there are other concepts that use different formulations of the PDE for training the network, one of them is the DeepRitz method. There, the energy formulation
of the problem is used.

Let us considere the following simple Laplace equation with a singularity at (0, 0):
\begin{align*}
    -\Delta u &= 1 \text{ in } \Omega, \\
    u &= 0 \text{ on } \partial \Omega,
\end{align*} 

with $\Omega=([-1, 1] \times [-1, 1]) \setminus ([0, 1] \times \{0\})$.

Then the energy functional for this problem is given by:
\begin{equation}
    E(v) = \int_\Omega \frac{1}{2}\|\nabla v\|^2 - v\text{d}x
\end{equation}
The solution $u$ of the strong formulation minimizes the energy functional, namely $E(u) \leq E(v)$ for all $v$ in the corresponding function space.

The DeepRitz method still tries to learn the solution $u$, but now minimizes $E$ in the training instead of using the strong formulation. To include the boundary condition, the functional also is extended:
\begin{equation}
    E(v) = \int_\Omega \frac{1}{2}\|\nabla v\|^2 - v \text{d}x + \int_{\partial\Omega} \frac{1}{2}\|v\|^2 \text{d}o
\end{equation}

The following cells, show the usage of DeepRitz in TorchPhysics.

In [66]:
# This block is for GPU selection. Please execute.
# import pathlib
# import os
# user = int(str(pathlib.Path().resolve())[22:24])
# os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

In [ ]:
import qewton

X = qewton.Variable('x', dim=2) 
U = qewton.Variable('u', dim=1)

# TODO: Implement the square [-1, 1] x [-1, 1] 
square = qewton.geometries.Rectangle
square_volume = square.volume()
square_surface = square.boundary.volume()

In [68]:
# Here we create the sampler. For DeepRitz we need to approximate the integral, therefore one usually needs more points
# then in the PINN approach to get a good estimate.
# But since in the energy functional the derivatives are of lower order, this generally does not lead to memory problems.
inner_sampler = qewton.GridSampler(square, n_points=100000) 

boundary_points = qewton.DEFAULT_DL_BACKEND.load_data('boundary_points.pt')
boundary_dataset = qewton.ArrayLikeDataSet(
    boundary_points, 
    qewton.DataConfiguration(qewton.BatchAxes(len(boundary_points)), qewton.FeatureAxes(X))
    )
boundary_sampler = qewton.DataLoader(boundary_dataset, batch_size=len(boundary_points))

In [ ]:
# TODO: Build a FCN with X as input, U as output, hidden layers of size 32 and 4 hidden layers
model = qewton.FCN

The transformation of the mathematical equations is handled similar to the PINN approach. For each integral in the energy functional, we define a constraint.

In [70]:
# The integral for the boundary: penalty/2 * \int u^2 do ≈ penalty/2 * |∂Ω| * mean(u^2)
penalty = 100.0

boundary_constraint = qewton.MSEConstraint(
    name="boundary", weight=0.5 * penalty * square_surface
)
boundary_pipeline = qewton.Graph()

with boundary_pipeline.tracker(0):
    x = boundary_sampler()
    u = model(x)
    boundary_constraint(u, 0.0)  # MSE against 0 is the mean of u^2

In [ ]:
# The integral for the inner part: \int 1/2 |∇u|^2 - u dx ≈ |Ω| * mean(1/2 |∇u|^2 - u)
def energy_density(u: U, x: X):
    u_x = u.gradient(x)
    return 0.5 * u_x.norm()**2 - u

energy_constraint = qewton.PINNConstraint(
    energy_density,
    reduction=qewton.bb.Mean(),
    weight=square_volume,
    name="energy",
)

# TODO: Following the implementation in the above cell, complete pipeline calls
energy_pipeline = qewton.Graph()
with energy_pipeline.tracker(0):
    ...
    x = x.track_gradient() # Track gradient for energy computation
    ...

In [ ]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=2500,
    lr_scheduler=qewton.optim.StepLR(step_size=800, gamma=0.5)
)

lbfgs_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.LBFGS(),
    lr=1.0,
    max_iterations=500,
    optimizer_args={"max_eval": 5}
)

# TODO: Collect all training phases, pipelines and constraints for the trainer
trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=,
    graphs=,
    training_objectives=,
    device=qewton.cuda(0),
)

trainer.run()

Optimization Phase 1:   0%|          | 1/2500 [00:00<00:33, 73.90it/s, loss=298]

Optimization Phase 2: 100%|██████████| 500/500 [00:52<00:00,  9.47it/s, loss=-0.176]


In [73]:
layout = energy_pipeline.visualize(model.output_ports[0], 
                             plot_type=qewton.viz.SurfacePlot, 
                             z=U)
qewton.viz.Figure(layout).show()